# Checkpoint 4: Hyperparameter Tuning of the Best Model (Random Forest)


## 1. Restoring the works

In [77]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split, KFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", 70)


In [78]:
df = pd.read_csv("house_sale.csv")
df

,id_x,rel_url,estate_rel_url_x,datetime_scrape_x,price,currency_x,location,attributes,city_when,city,day_x,hour_x,repair,vip,featured,products_label,bill_of_sale,mortgage,img_url,id_y,estate_id,estate_rel_url_y,datetime_scrape_y,description,unit_price,total_price,currency_y,owner_name,owner_title,shop_name,shop_title,address,lat,lng,updated,views,day_y,hour_y,estate_details_id_x,Binanın növü,Kateqoriya,Mərtəbə,Otaq sayı,Sahə,Torpaq sahəsi,Təmir,Çıxarış,İpoteka,estate_details_id_y,estate_rel_url,extra_info
0,5df36281-6dc6-4d5d-89a7-5fcfa86f7608,/alqi-satqi?page=174,/items/4521724,2024-10-05 22:07:37.60613+00,499999.0,AZN,Səbail r.,"4 otaqlı, 145 m², 7/9 mərtəbə","Bakı, dünən 23:52",bakı,05.10.2024,23:52,Təmirli,vipped,featured,NaN,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,92e82ea2-e1f3-4c2d-a284-151efd99281e,5df36281-6dc6-4d5d-89a7-5fcfa86f7608,/items/4521724,2024-10-05 22:14:10.089116+00,"Səbail Rayonu, İzzət Nəbiyev küçəsi, Fəxri Xiy...",3 450 AZN/m²,499999.0,AZN,Kamran,mülkiyyətçi,NaN,NaN,İzzət Nəbiyev küç.,40.358817,49.824092,yeniləndi: dünən 23:52,1155,05.10.2024,23:52,92e82ea2-e1f3-4c2d-a284-151efd99281e,NaN,Köhnə tikili,7 / 9,4.0,145 m²,NaN,var,var,NaN,92e82ea2-e1f3-4c2d-a284-151efd99281e,/items/4521724,Şəhidlər xiyabanı * Dağüstü parkı * Səbail r.
1,883e20f0-8872-49a5-8b4b-63b8301b5f8f,/alqi-satqi?page=250,/items/4669294,2024-10-05 22:07:37.60613+00,77000.0,AZN,Biləcəri q.,"4 otaqlı, 90 m²","Bakı, dünən 23:56",bakı,05.10.2024,23:56,Təmirli,NaN,NaN,NaN,NaN,NaN,https://bina.azstatic.com/uploads/f460x345/202...,505eaf81-6bc8-4094-9b00-aa82066548ee,883e20f0-8872-49a5-8b4b-63b8301b5f8f,/items/4669294,2024-10-05 22:14:10.089116+00,"Biləcəridə Abidəyə yaxin 91,92,202 saylı marşr...",NaN,77000.0,AZN,Dasinmaz Emlak,vasitəçi (agent),NaN,NaN,Biləcəri qəs.,40.420897,49.807035,yeniləndi: 04 oktyabr 2024,218,04.10.2024,NaN,505eaf81-6bc8-4094-9b00-aa82066548ee,NaN,Həyət evi/Bağ evi,NaN,4.0,90 m²,1.3 sot,var,yoxdur,NaN,505eaf81-6bc8-4094-9b00-aa82066548ee,/items/4669294,Binəqədi r.* Biləcəri q.
2,55c36fb1-a3af-476e-ba17-a81f6795be8d,/alqi-satqi?page=250,/items/4669293,2024-10-05 22:07:37.60613+00,92000.0,AZN,İnşaatçılar m.,"3 otaqlı, 60 m²","Bakı, dünən 23:55",bakı,05.10.2024,23:55,Təmirli,NaN,NaN,NaN,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,fa63b201-999d-43b5-a61a-778d9d79a6c6,55c36fb1-a3af-476e-ba17-a81f6795be8d,/items/4669293,2024-10-05 22:14:10.089116+00,Salam əleykum. \nİnşaatçılar metrosuna yaxın m...,NaN,92000.0,AZN,Məhəmməd,vasitəçi (agent),NaN,NaN,Mirzə Cabbar Məmmədzadə küç.,40.390293,49.802656,yeniləndi: 04 oktyabr 2024,190,04.10.2024,NaN,fa63b201-999d-43b5-a61a-778d9d79a6c6,NaN,Həyət evi/Bağ evi,NaN,3.0,60 m²,0.1 sot,var,var,NaN,fa63b201-999d-43b5-a61a-778d9d79a6c6,/items/4669293,İnşaatçılar m.* Yasamal r.
3,acf1aa8d-a46a-40f5-b6f2-f7a449569337,/alqi-satqi?page=250,/items/4647811,2024-10-05 22:07:37.60613+00,95000.0,AZN,Qaraçuxur q.,130 m²,"Bakı, dünən 23:55",bakı,05.10.2024,23:55,Təmirli,vipped,featured,NaN,Çıxarış var,İpoteka var,https://bina.azstatic.com/uploads/f460x345/202...,5db56980-05cc-4925-b55b-f58fb3f4d2b6,acf1aa8d-a46a-40f5-b6f2-f7a449569337,/items/4647811,2024-10-05 22:14:10.089116+00,Barter maraqlidir üstünlük maşina verilir\nHər...,NaN,95000.0,AZN,Elçin,mülkiyyətçi,NaN,NaN,Qaraçuxur qəs.,40.393614,49.981553,yeniləndi: 04 oktyabr 2024,1314,04.10.2024,NaN,5db56980-05cc-4925-b55b-f58fb3f4d2b6,NaN,Obyekt,NaN,NaN,130 m²,NaN,var,var,var,5db56980-05cc-4925-b55b-f58fb3f4d2b6,/items/4647811,Suraxanı r.* Qaraçuxur q.
4,22d840df-9283-4112-bc71-7432511fc776,/alqi-satqi?page=250,/items/4638863,2024-10-05 22:07:37.60613+00,220000.0,AZN,Əhmədli m.,"3 otaqlı, 100 m², 15/16 mərtəbə","Bakı, dünən 23:52",bakı,05.10.2024,23:52,Təmirli,NaN,NaN,Agentlik,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,d71cf9a9-86dd-4162-b540-6252a8659a09,22d840df-9283-4112-bc71-7432511fc776,/items/4638863,2024-10-05 22:14:10.089116+00,Əhmədli qəs. Qaçaq Nəbi küçəsi 3 otaga duzelm...

In [79]:
df_clean = df.copy()
df_clean = df_clean.drop(columns = ["Binanın növü", "hour_y"])

In [80]:
df_clean["bill_of_sale_flag"] = df_clean["Çıxarış"].str.strip().str.lower().map({"var": 1, "yoxdur": 0})
df_clean = df_clean.drop(columns=["featured", "vip", "mortgage", "İpoteka", "bill_of_sale", "Çıxarış", "repair"])
df_clean["Təmir"] = df_clean["Təmir"].fillna("Unknown")

In [81]:
def seller_type(row):
    if pd.notnull(row["shop_name"]):
        return "Agency"
    elif pd.notnull(row["owner_name"]):
        return "Individual"
    return "Unknown"


In [82]:
df_clean["seller_type"] = df_clean.apply(seller_type, axis = 1)

In [83]:
def parse_area(val):
    if pd.isnull(val):
        return np.nan
    val = val.strip()
    if "sot" in val:
        return float(val.replace("sot", "").strip().replace(",", ".")) * 100
    return float(val.replace("m²", "").strip().replace(" ", ""))

In [84]:
df_clean["Sahə_numeric"] = df_clean["Sahə"].apply(parse_area)

In [85]:
applicable_categories = ["Köhnə tikili", "Yeni tikili", "Ofis", "Həyət evi/Bağ evi"]
mask = df_clean["Kateqoriya"].isin(applicable_categories)
medians = df_clean.loc[mask].groupby("Kateqoriya")["Otaq sayı"].median()

In [86]:
for cat, med in medians.items():
    idx = df_clean[(df_clean["Kateqoriya"] == cat) & (df_clean["Otaq sayı"].isnull())].index
    df_clean.loc[idx, "Otaq sayı"] = med

In [87]:
df_clean = df_clean[df_clean["price"] >= 1000].copy()

In [88]:
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

In [89]:
def cap_grouped(df, col, group_col="Kateqoriya", k=1.5):
    capped = df[col].copy()
    for cat, group in df.groupby(group_col):
        low, high = iqr_bounds(group[col].dropna(), k)
        idx = group.index
        capped.loc[idx[df.loc[idx, col] < low]] = low
        capped.loc[idx[df.loc[idx, col] > high]] = high
    return capped


In [90]:
df_clean["price_capped"] = cap_grouped(df_clean, "price")
df_clean["Sahə_numeric_capped"] = cap_grouped(df_clean, "Sahə_numeric")

In [91]:
low_v, high_v = iqr_bounds(df_clean["views"])
df_clean["views_capped"] = df_clean["views"].clip(low_v, high_v)

In [92]:
floor_split = df_clean["Mərtəbə"].str.split("/", expand = True)
df_clean["floor_current"] = pd.to_numeric(floor_split[0].str.strip(), errors = "coerce")
df_clean["floor_total"] = pd.to_numeric(floor_split[1].str.strip(), errors = "coerce")
df_clean["floor_ratio"] = (df_clean["floor_current"] / df_clean["floor_total"]).round(2)

In [93]:
def loc_type(loc):
    if pd.isnull(loc):
        return "Unknown"
    match = re.search(r"\s(r|m|q)\.$", loc.strip())
    mapping = {"r": "District center", "m": "Near metro", "q": "Settlement"}
    return mapping.get(match.group(1), "Other") if match else "Other"

In [94]:
df_clean["location_type"] = df_clean["location"].apply(loc_type)

In [95]:
feature_columns = ["Sahə_numeric_capped", "Otaq sayı", "views_capped", "floor_ratio",
                    "Kateqoriya", "location_type", "seller_type", "Təmir", "city"]
target_column = "price_capped"

In [96]:
df_model = df_clean[feature_columns + [target_column]].copy()
X = df_model[feature_columns]
y = df_model[target_column]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [97]:
numeric_features = ["Sahə_numeric_capped", "Otaq sayı", "views_capped", "floor_ratio"]
categorical_features = ["Kateqoriya", "location_type", "seller_type", "Təmir", "city"]

In [98]:
numeric_pipeline = Pipeline(steps = [
    ("impute", SimpleImputer(strategy = "median")),
    ("scale", StandardScaler()),
])
categorical_pipeline = Pipeline(steps = [
    ("impute", SimpleImputer(strategy = "constant", fill_value = "Unknown")),
    ("encode", OneHotEncoder(handle_unknown = "ignore")),
])
preprocessor = ColumnTransformer(transformers = [
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

In [99]:
random_forest_model = Pipeline(steps = [
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators = 150,
                                    max_depth = 14,
                                    random_state = 42,
                                    n_jobs = -1)),
])

In [100]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (80612, 9)
X_test: (20153, 9)


## 2. Baseline to beat

This is the untuned Random Forest score from Checkpoint 3 (same 5-fold CV, same metrics), the number the tuned model needs to improve on.


In [101]:
baseline_kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
baseline_scores = cross_validate(
    random_forest_model, X_train, y_train, cv=baseline_kf,
    scoring = {"MAE": "neg_mean_absolute_error",
               "RMSE": "neg_root_mean_squared_error",
               "R2": "r2"},
    n_jobs = 1,
)

In [102]:
baseline_mae = -baseline_scores["test_MAE"]
baseline_r2 = baseline_scores["test_R2"]
print(f"Baseline Random Forest -> MAE: {baseline_mae.mean():,.0f} ± {baseline_mae.std():,.0f} | "
      f"R2: {baseline_r2.mean():.3f} ± {baseline_r2.std():.3f}")


Baseline Random Forest -> MAE: 62,213 ± 1,224 | R2: 0.767 ± 0.014


## 3. Hyperparameter search

RandomizedSearchCV is used instead of a full grid search, with 4 hyperparameters and several values each, a full grid would mean way too many combinations to fit in reasonable time. RandomizedSearchCV samples a fixed number of random combinations instead, which gets most of the benefit of a full search at a fraction of the cost.

To keep the search itself fast, it uses a lighter 3-fold CV internally (search_cv), that's just for comparing candidate hyperparameters quickly. Once the best combination is found, it gets validated properly below using the exact same 5-fold CV, so the final reported number is directly comparable to the baseline.

The search is scored on MAE, matching the main business metric.


In [103]:
param_distributions = {
    "model__n_estimators": [100, 150, 200, 250],
    "model__max_depth": [8, 12, 16, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5],
}

In [104]:
search_cv = KFold(n_splits = 3,
                  shuffle = True,
                  random_state = 42)

In [105]:
random_search = RandomizedSearchCV(
    estimator = random_forest_model,
    param_distributions = param_distributions,
    n_iter = 10,
    scoring = "neg_mean_absolute_error",
    cv = search_cv,
    random_state = 42,
    n_jobs = 1,
)

In [107]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=KFold(n_splits=3, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('impute',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scale',
                                                                                                StandardScaler())]),
                                                                               ['Sahə_numeric_capped',
                                                                                'Otaq '
                                                                                'sayı',
                                                                                'views_capped',
                                                                                'floor_ratio']),
                                                                              ('cat',
                                                                               Pipeline(steps=[('impute',
                                                                                                SimpleImput...
                                                                                'seller_type',
                                                                                'Təmir',
                                                                                'city'])])),
                                             ('model',
                                              RandomForestRegressor(max_depth=14,
                                                                    n_estimators=150,
                                                                    n_jobs=-1,
                                                                    random_state=42))]),
                   n_jobs=1,
                   param_distributions={'model__max_depth': [8, 12, 16, 20],
                                        'model__max_features': ['sqrt', 'log2',
                                                                0.5],
                                        'model__min_samples_leaf': [1, 2, 4, 8],
                                        'model__n_estimators': [100, 150, 200,
                                                                250]},
                   random_state=42, scoring='neg_mean_absolute_error')

In [108]:
print("Best params found:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

Best params found:
  model__n_estimators: 150
  model__min_samples_leaf: 1
  model__max_features: 0.5
  model__max_depth: 20


In [109]:
print(f"Best CV MAE during search: {-random_search.best_score_:,.0f}")

Best CV MAE during search: 58,301


## 4. Validate the tuned model with the same 5-fold CV as the baseline

This is the fair, apples-to-apples comparison: same KFold(5, random_state=42), same metrics, only the hyperparameters changed.


In [110]:
tuned_model = random_search.best_estimator_

In [112]:
tuned_scores = cross_validate(
    tuned_model, X_train, y_train, cv = baseline_kf,
    scoring = {"MAE": "neg_mean_absolute_error",
               "RMSE": "neg_root_mean_squared_error",
               "R2": "r2"},
    n_jobs = 1,
)

In [113]:
tuned_mae = -tuned_scores["test_MAE"]
tuned_rmse = -tuned_scores["test_RMSE"]
tuned_r2 = tuned_scores["test_R2"]

In [114]:
comparison_df = pd.DataFrame([
    {"model": "Random Forest (baseline)", "MAE": f"{baseline_mae.mean():,.0f} ± {baseline_mae.std():,.0f}",
     "RMSE": f"{(-baseline_scores['test_RMSE']).mean():,.0f} ± {(-baseline_scores['test_RMSE']).std():,.0f}",
     "R2": f"{baseline_r2.mean():.3f} ± {baseline_r2.std():.3f}"},
    {"model": "Random Forest (tuned)", "MAE": f"{tuned_mae.mean():,.0f} ± {tuned_mae.std():,.0f}",
     "RMSE": f"{tuned_rmse.mean():,.0f} ± {tuned_rmse.std():,.0f}",
     "R2": f"{tuned_r2.mean():.3f} ± {tuned_r2.std():.3f}"},
])

In [115]:
comparison_df

,model,MAE,RMSE,R2
0,Random Forest (baseline),"62,213 ± 1,224","136,356 ± 6,795",0.767 ± 0.014
1,Random Forest (tuned),"57,159 ± 1,149","134,477 ± 5,981",0.773 ± 0.012


## 5. Written interpretation

the table above compares the untuned baseline to the tuned model, both scored with the identical 5-fold CV setup, so any difference reflects the hyperparameters, not a change in evaluation method.

If the tuned MAE is meaningfully lower, the search found a genuinely better setting. If the improvement is small or within the std range of the baseline, that's still a useful finding.

it means the untuned defaults were already close to as good as this model family can get on this data, and further gains would more likely come from better features than from more tuning.